# Ghost Job Detection Pipeline — XGBoost + SHAP + GMM

**Course:** COMM 486I — AI in Finance (UBC Sauder)

### Three outputs
| # | Output | Method |
|---|--------|--------|
| 1 | **Predicted fill rate** per company | XGBoost regression → predicted inflows / open postings |
| 2 | **Actual fill rate** per company | Empirical: actual inflows / open postings |
| 3 | **Posting-level ghost score** | SHAP feature weights from trained model |

### Sections
1. Setup & Data Loading
2. Target Variable: Quarterly Inflows
3. Feature Engineering: Postings
4. Feature Engineering: Company
5. Feature Matrix Assembly
6. Model Training & Evaluation
7. SHAP Analysis
8. Three Measures Comparison
9. Posting-Level Ghost Score
10. GMM Validation
11. Outputs & Visualization

## 1  Setup & Data Loading

In [ ]:
# Install packages (Colab)
!pip install -q polars pyarrow scikit-learn xgboost shap matplotlib seaborn

import polars as pl
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"polars {pl.__version__}")
print("All packages ready.")

In [ ]:
# ── DATA PATHS — edit these before running ────────────────────────────────
# Colab  : set DATA_DIR to your Google Drive folder containing the Revelio parquet files
#          e.g. '/content/drive/MyDrive/revelio'
# Local  : set DATA_DIR to the absolute path of your local data folder
#          e.g. '/home/user/data/revelio'  or  r'C:\\Users\\you\\data\\revelio'
DATA_DIR   = '/content/drive/MyDrive/revelio'   # ← change this
OUTPUT_DIR_STR = '/content/drive/MyDrive/ghost_job_output'  # ← change this

# ── Mount Google Drive (Colab only — comment out if running locally) ──────
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # not running in Colab

from pathlib import Path
BASE       = Path(DATA_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR_STR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# NOTE: Revelio Labs folder names use 'positings' (typo in source data), not 'postings'
PATHS = {
    'company_ref': BASE / 'revelio_company_ref',
    'positions':   BASE / 'revelio_individual_position',
    'postings':    BASE / 'revelio_positings_unified_individual',
}

print(f"Data dir   : {BASE}")
print(f"Output dir : {OUTPUT_DIR}")
for name, path in PATHS.items():
    n = len(list(path.glob('*.parquet'))) if path.exists() else 0
    status = f'{n} files' if path.exists() else 'MISSING ⚠'
    print(f"  {name:<16} {status}")


In [ ]:
# ── Load company universe ──────────────────────────────────────────────────
US_EXCHANGES = [
    'NASDAQ', 'New York Stock Exchange', 'US OTC', 'NYSE American', 'NYSE Arca'
]

companies = (
    pl.scan_parquet(PATHS['company_ref'] / '*.parquet', hive_partitioning=False)
    .select(['rcid', 'company', 'ticker', 'exchange_name', 'naics_code'])
    .filter(
        pl.col('ticker').is_not_null()
        & (pl.col('ticker') != '')
        & pl.col('exchange_name').is_in(US_EXCHANGES)
    )
    .collect()
    .unique(subset=['rcid'], keep='first')
)

target_rcids = companies['rcid'].unique().to_list()
print(f"Target US-listed firms with tickers : {len(target_rcids):,}")
print(f"Unique tickers                       : {companies['ticker'].n_unique():,}")
print()
display(companies.group_by('exchange_name').len().sort('len', descending=True))

## 2  Target Variable: Quarterly Inflows

- **actual_inflows**: distinct `user_id`s whose `startdate` falls in a given quarter
- **headcount**: running total of employees (cumsum of inflows minus outflows)
- These give a table: `rcid | quarter | actual_inflows | outflows | headcount`

In [ ]:
# ── Helper: date → 'YYYY-QN' string ───────────────────────────────────────
def to_quarter(date_col: str) -> pl.Expr:
    """Return 'YYYY-QN' string from a parsed Date column."""
    return (
        pl.col(date_col).dt.year().cast(pl.Utf8)
        + '-Q'
        + pl.col(date_col).dt.quarter().cast(pl.Utf8)
    )

def quarter_sort_key(q_col: str) -> pl.Expr:
    """Integer sort key for 'YYYY-QN' strings (for correct temporal ordering)."""
    return (
        pl.col(q_col).str.slice(0, 4).cast(pl.Int32) * 4
        + pl.col(q_col).str.slice(6, 1).cast(pl.Int32) - 1
    )

In [ ]:
# ── Load positions ─────────────────────────────────────────────────────────
print("Loading individual positions (this may take several minutes)...")
positions_raw = (
    pl.scan_parquet(PATHS['positions'] / '*.parquet', hive_partitioning=False)
    .filter(pl.col('rcid').is_in(target_rcids))
    .select(['rcid', 'user_id', 'startdate', 'enddate'])
    .collect()
)
print(f"Loaded {len(positions_raw):,} position records "
      f"for {positions_raw['rcid'].n_unique():,} firms")

# Parse dates
positions_raw = positions_raw.with_columns([
    pl.col('startdate').str.to_date('%Y-%m-%d', strict=False).alias('start_dt'),
    pl.col('enddate').str.to_date('%Y-%m-%d', strict=False).alias('end_dt'),
])

null_start = positions_raw['start_dt'].null_count()
null_end   = positions_raw['end_dt'].null_count()
print(f"Null startdate: {null_start:,}  |  Null enddate (still active): {null_end:,} "
      f"({null_end/len(positions_raw)*100:.1f}%)")

In [ ]:
# ── Inflows: unique users starting per firm-quarter ────────────────────────
inflows = (
    positions_raw
    .filter(pl.col('start_dt').is_not_null())
    .with_columns(to_quarter('start_dt').alias('quarter'))
    .group_by(['rcid', 'quarter'])
    .agg(pl.col('user_id').n_unique().alias('actual_inflows'))
)

# ── Outflows: unique users leaving per firm-quarter ────────────────────────
outflows = (
    positions_raw
    .filter(pl.col('end_dt').is_not_null())
    .with_columns(to_quarter('end_dt').alias('quarter'))
    .group_by(['rcid', 'quarter'])
    .agg(pl.col('user_id').n_unique().alias('outflows'))
)

# ── Running headcount via cumulative (inflows − outflows) ──────────────────
# NOTE: headcount is relative (starts from 0 at first observed quarter per firm).
# It captures *growth* correctly, which is all we need for headcount_growth_4q.
headcount_df = (
    inflows
    .join(outflows, on=['rcid', 'quarter'], how='left')
    .with_columns(pl.col('outflows').fill_null(0))
    .with_columns(quarter_sort_key('quarter').alias('_qkey'))
    .sort(['rcid', '_qkey'])
    .with_columns(
        (pl.col('actual_inflows') - pl.col('outflows'))
        .cum_sum()
        .over('rcid')
        .alias('headcount')
    )
    .drop('_qkey')
)

print(f"Firm-quarters with inflow data : {len(headcount_df):,}")
print(f"Unique firms                   : {headcount_df['rcid'].n_unique():,}")
print(f"Quarter range                  : "
      f"{headcount_df['quarter'].sort().item(0)} → "
      f"{headcount_df['quarter'].sort()[-1]}")

# Sanity check
zero_firms = (
    headcount_df.group_by('rcid')
    .agg(pl.col('actual_inflows').sum())
    .filter(pl.col('actual_inflows') == 0)
)
print(f"\nSanity — firms with zero total inflows: {len(zero_firms):,} (will be excluded in Step 5)")

## 3  Feature Engineering: Postings

Aggregate unified postings to firm-quarter level.
Key features: posting volume, duration, still-active rate, salary disclosure, remote rate, role diversity.

In [ ]:
# ── Load unified postings ──────────────────────────────────────────────────
POSTING_COLS = [
    'job_id', 'rcid', 'salary', 'post_date', 'remove_date',
    'remote_type', 'expected_hires', 'job_category', 'role_k50',
]

print("Loading unified postings (may take 5–10 min from Drive)...")
postings_raw = (
    pl.scan_parquet(PATHS['postings'] / '*.parquet', hive_partitioning=False)
    .filter(pl.col('rcid').is_in(target_rcids))
    .select(POSTING_COLS)
    .collect()
)
print(f"Loaded {len(postings_raw):,} postings "
      f"for {postings_raw['rcid'].n_unique():,} firms")

# Parse dates and cast numerics
postings_raw = postings_raw.with_columns([
    pl.col('post_date').str.to_date('%Y-%m-%d', strict=False).alias('post_dt'),
    pl.col('remove_date').str.to_date('%Y-%m-%d', strict=False).alias('remove_dt'),
    pl.col('salary').cast(pl.Float64, strict=False),
    pl.col('expected_hires').cast(pl.Float64, strict=False),
])

# Duration (days) and quarter
postings_raw = postings_raw.with_columns([
    (pl.col('remove_dt') - pl.col('post_dt')).dt.total_days().alias('duration_days'),
    to_quarter('post_dt').alias('quarter'),
])

print(f"\nDate range: {postings_raw['post_dt'].min()} → {postings_raw['post_dt'].max()}")
print(f"Null post_date     : {postings_raw['post_dt'].null_count():,}")
print(f"Still active (null remove_date): {postings_raw['remove_dt'].null_count():,} "
      f"({postings_raw['remove_dt'].null_count()/len(postings_raw)*100:.1f}%)")
print(f"Salary disclosed   : {postings_raw['salary'].is_not_null().sum():,} "
      f"({postings_raw['salary'].is_not_null().sum()/len(postings_raw)*100:.1f}%)")

In [ ]:
# ── Aggregate to firm-quarter ──────────────────────────────────────────────
posting_features = (
    postings_raw
    .filter(pl.col('post_dt').is_not_null())
    .group_by(['rcid', 'quarter'])
    .agg([
        pl.len().alias('num_postings'),
        # Duration
        pl.col('duration_days').mean().alias('avg_duration'),
        pl.col('duration_days').median().alias('median_duration'),
        # Still-active rate (null remove_date = never removed)
        pl.col('remove_dt').is_null().mean().alias('still_active_rate'),
        # Salary
        pl.col('salary').is_not_null().mean().alias('salary_disclosure_rate'),
        pl.col('salary').mean().alias('avg_salary'),
        # Expected hires
        pl.col('expected_hires').sum().alias('total_expected_hires'),
        pl.col('expected_hires').mean().alias('avg_expected_hires'),
        # Role diversity: unique roles / total postings
        (pl.col('role_k50').n_unique() / pl.len()).alias('role_diversity'),
        # Remote rate
        (pl.col('remote_type') == 'Remote').mean().alias('remote_rate'),
    ])
)

print(f"Posting feature matrix: {posting_features.shape}")
print(f"Firms: {posting_features['rcid'].n_unique():,}")
print(f"Quarter range: {posting_features['quarter'].sort().item(0)} → "
      f"{posting_features['quarter'].sort()[-1]}")
display(posting_features.describe())

## 4  Feature Engineering: Company

- `headcount` — running headcount from Step 2
- `headcount_growth_4q` — % change vs 4 quarters ago (YoY)
- `log_headcount` — size control
- `naics_2digit` — industry sector (first 2 digits of NAICS code)

In [ ]:
# ── Headcount growth features ──────────────────────────────────────────────
company_features = (
    headcount_df
    .with_columns(quarter_sort_key('quarter').alias('_qkey'))
    .sort(['rcid', '_qkey'])
    .with_columns([
        pl.col('headcount').shift(4).over('rcid').alias('_hc_4q_ago'),
    ])
    .with_columns([
        (
            (pl.col('headcount') - pl.col('_hc_4q_ago'))
            / pl.col('_hc_4q_ago').abs().clip(lower_bound=1)
        ).alias('headcount_growth_4q'),
        pl.col('headcount').log1p().alias('log_headcount'),
    ])
    .drop(['_qkey', '_hc_4q_ago'])
)

# ── NAICS 2-digit (industry sector) ───────────────────────────────────────
companies_with_naics = companies.with_columns(
    pl.col('naics_code')
    .cast(pl.Utf8, strict=False)
    .str.slice(0, 2)
    .alias('naics_2digit')
)

print("Company features ready:")
display(company_features.select(['headcount', 'headcount_growth_4q', 'log_headcount']).describe())
print(f"\nNAICS 2-digit distribution (top 10 sectors):")
display(
    companies_with_naics.group_by('naics_2digit').len()
    .sort('len', descending=True).head(10)
)

## 5  Feature Matrix Assembly

Join posting features + company features + actual inflows.
**Time-shift target**: features from quarter Q predict inflows in Q+1 (no leakage).
Drop firm-quarters with < 3 postings (too noisy).

In [ ]:
# ── Join all tables ────────────────────────────────────────────────────────
feature_matrix = (
    posting_features
    .join(company_features, on=['rcid', 'quarter'], how='left')
    .join(
        companies_with_naics.select(['rcid', 'company', 'ticker', 'naics_2digit']),
        on='rcid', how='left'
    )
)

# ── Filter: drop firm-quarters with too few postings ──────────────────────
before = len(feature_matrix)
feature_matrix = feature_matrix.filter(pl.col('num_postings') >= 3)
print(f"Dropped {before - len(feature_matrix):,} firm-quarters with < 3 postings")

# ── Time-shift: next-quarter inflows as target ─────────────────────────────
feature_matrix = (
    feature_matrix
    .with_columns(quarter_sort_key('quarter').alias('_qkey'))
    .sort(['rcid', '_qkey'])
    .with_columns(
        pl.col('actual_inflows').shift(-1).over('rcid').alias('actual_inflows_next_q')
    )
    .drop('_qkey')
)

# Drop the last quarter per firm (no future to predict) and any null targets
feature_matrix = feature_matrix.filter(pl.col('actual_inflows_next_q').is_not_null())

# ── Encode NAICS as integer for XGBoost ───────────────────────────────────
feature_matrix = feature_matrix.with_columns(
    pl.col('naics_2digit').cast(pl.Int32, strict=False).fill_null(-1).alias('naics_int')
)

print(f"\nFeature matrix: {feature_matrix.shape}")
print(f"Firms          : {feature_matrix['rcid'].n_unique():,}")
print(f"Quarter range  : {feature_matrix['quarter'].sort().item(0)} → "
      f"{feature_matrix['quarter'].sort()[-1]}")

# Null audit
null_counts = {c: feature_matrix[c].null_count() for c in feature_matrix.columns}
noisy = {k: v for k, v in null_counts.items() if v > 0}
if noisy:
    print("\nColumns with nulls:")
    for col, cnt in sorted(noisy.items(), key=lambda x: -x[1]):
        print(f"  {col:<30} {cnt:>8,} ({cnt/len(feature_matrix)*100:.1f}%)")

# Sanity: firms with zero inflows across all quarters
zero_firms = (
    feature_matrix.group_by('rcid')
    .agg(pl.col('actual_inflows_next_q').sum().alias('tot'))
    .filter(pl.col('tot') == 0)
)
print(f"\nSanity — firms with zero total future inflows: {len(zero_firms):,}")
print("  (retained — they may be genuinely ghost-like)")

In [ ]:
# ── Save feature matrix ────────────────────────────────────────────────────
feature_matrix.write_parquet(OUTPUT_DIR / 'firm_quarter_features.parquet')
print(f"Saved firm_quarter_features.parquet  ({len(feature_matrix):,} rows)")

## 6  Model Training & Evaluation

**Target**: `actual_inflows_next_q` (raw count)
**Split** (by firm):
- Train: 80% of firms (random, seed=42)
- Val: 10% of firms
- Test: 10% of firms

Two models:
1. **XGBoost** with early stopping
2. **Ridge regression** as baseline

In [ ]:
FEATURE_COLS = [
    # Posting features
    'num_postings', 'avg_duration', 'median_duration', 'still_active_rate',
    'salary_disclosure_rate', 'avg_salary', 'total_expected_hires',
    'avg_expected_hires', 'role_diversity', 'remote_rate',
    # Company features
    'headcount', 'log_headcount', 'headcount_growth_4q',
    'naics_int',
]
TARGET_COL = 'actual_inflows_next_q'

# ─────────────────────split─────────────────────────────────
from sklearn.model_selection import train_test_split

# Get unique firms and split 80/20
all_rcids = feature_matrix['rcid'].unique().to_list()
train_rcids, test_rcids = train_test_split(all_rcids, test_size=0.20, random_state=42)
val_rcids,  test_rcids  = train_test_split(test_rcids, test_size=0.50, random_state=42)
# now: 80% train, 10% val, 10% test

train_df = feature_matrix.filter(pl.col('rcid').is_in(train_rcids))
val_df   = feature_matrix.filter(pl.col('rcid').is_in(val_rcids))
test_df  = feature_matrix.filter(pl.col('rcid').is_in(test_rcids))

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"{name:<6} {len(df):>7,} rows  {df['rcid'].n_unique():,} firms")

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"{name:<6} {len(df):>7,} rows  "
          f"{df['rcid'].n_unique():,} firms  "
          f"{df['quarter'].min()} → {df['quarter'].max()}")

def to_xy(df):
    """Polars DataFrame → (X_float32, y_float32), filling nulls with column mean."""
    X = df.select(FEATURE_COLS).fill_null(strategy='mean').to_numpy().astype('float32')
    y = df[TARGET_COL].to_numpy().astype('float32')
    return X, y

X_train, y_train = to_xy(train_df)
X_val,   y_val   = to_xy(val_df)
X_test,  y_test  = to_xy(test_df)
print(f"\nFeature shapes — train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")

In [ ]:
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

# ── XGBoost ────────────────────────────────────────────────────────────────
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=30,
    eval_metric='mae',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f"  Best iteration: {xgb_model.best_iteration}")

# ── Ridge baseline ─────────────────────────────────────────────────────────
print("Training Ridge regression (baseline)...")
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
ridge = Ridge(alpha=10.0)
ridge.fit(X_train_s, y_train)

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
from sklearn.metrics import r2_score, mean_absolute_error

def eval_model(model, Xtr, ytr, Xv, yv, Xte, yte, label, scaler_=None):
    if scaler_ is not None:
        Xtr, Xv, Xte = scaler_.transform(Xtr), scaler_.transform(Xv), scaler_.transform(Xte)
    preds = {}
    for split, X, y in [('Train', Xtr, ytr), ('Val', Xv, yv), ('Test', Xte, yte)]:
        p = np.maximum(model.predict(X), 0)
        preds[split] = p
        print(f"  {label:<18} {split:<6}  R²={r2_score(y,p):+.4f}  MAE={mean_absolute_error(y,p):.2f}")
    return preds

print(f"{'─'*60}")
xgb_preds  = eval_model(xgb_model, X_train, y_train, X_val, y_val,
                         X_test, y_test, 'XGBoost')
print(f"{'─'*60}")
ridge_preds = eval_model(ridge, X_train, y_train, X_val, y_val,
                          X_test, y_test, 'Ridge', scaler_=scaler)
print(f"{'─'*60}")

# Store test predictions
xgb_test_pred   = xgb_preds['Test']
ridge_test_pred = ridge_preds['Test']

In [ ]:
# ── Feature importance (built-in gain) ────────────────────────────────────
fi = (
    pl.DataFrame({
        'feature':    FEATURE_COLS,
        'importance': xgb_model.feature_importances_,
    })
    .sort('importance', descending=True)
)
display(fi)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi['feature'].to_list()[::-1], fi['importance'].to_list()[::-1], color='steelblue')
ax.set_xlabel('Gain Importance')
ax.set_title('XGBoost Feature Importance (Gain)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EXTENDED MODEL ACCURACY DIAGNOSTICS ────────────────────────────────────
# Add this cell after Section 6 (feature importance), before SHAP

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error

print("=" * 60)
print("EXTENDED MODEL ACCURACY DIAGNOSTICS")
print("=" * 60)

# ── 1. Residual stats per split ────────────────────────────────────────────
for split_name, X, y in [('Train', X_train, y_train),
                           ('Val',   X_val,   y_val),
                           ('Test',  X_test,  y_test)]:
    p = np.maximum(xgb_model.predict(X), 0)
    residuals = y - p
    print(f"\n{split_name}:")
    print(f"  R²                  : {r2_score(y, p):+.4f}")
    print(f"  MAE                 : {mean_absolute_error(y, p):.2f}")
    print(f"  Median AE           : {median_absolute_error(y, p):.2f}")
    print(f"  Mean residual (bias): {residuals.mean():.2f}  "
          f"({'over-predicts' if residuals.mean() < 0 else 'under-predicts'})")
    print(f"  Residual std        : {residuals.std():.2f}")
    print(f"  % within ±5 hires   : {(np.abs(residuals) <= 5).mean()*100:.1f}%")
    print(f"  % within ±10 hires  : {(np.abs(residuals) <= 10).mean()*100:.1f}%")
    # Naive baseline: always predict the training mean
    naive_pred = np.full_like(y, y_train.mean())
    print(f"  Naive baseline R²   : {r2_score(y, naive_pred):+.4f}  "
          f"(always predict train mean = {y_train.mean():.1f})")

# ── 2. Spearman rank correlation (ranking quality) ─────────────────────────
from scipy.stats import spearmanr, pearsonr
print("\n── Rank-ordering quality (Spearman ρ) ──")
print("   (1.0 = perfect rank order, 0 = random)")
for split_name, X, y in [('Train', X_train, y_train),
                           ('Val',   X_val,   y_val),
                           ('Test',  X_test,  y_test)]:
    p = np.maximum(xgb_model.predict(X), 0)
    rho, pval = spearmanr(y, p)
    r, _      = pearsonr(y, p)
    print(f"  {split_name:<6}  Spearman ρ = {rho:.4f}  Pearson r = {r:.4f}")

# ── 3. Calibration check — predicted vs actual by decile ──────────────────
print("\n── Calibration: mean actual vs mean predicted by decile (TEST set) ──")
p_test = np.maximum(xgb_model.predict(X_test), 0)
decile = np.percentile(p_test, np.arange(0, 110, 10))
print(f"  {'Decile':<10} {'Mean Predicted':>15} {'Mean Actual':>12} {'Ratio':>8}")
for i in range(len(decile) - 1):
    mask = (p_test >= decile[i]) & (p_test < decile[i+1])
    if mask.sum() == 0:
        continue
    mean_pred   = p_test[mask].mean()
    mean_actual = y_test[mask].mean()
    ratio = mean_actual / mean_pred if mean_pred > 0 else float('nan')
    print(f"  D{i+1:<9} {mean_pred:>15.2f} {mean_actual:>12.2f} {ratio:>8.2f}x")

# ── 4. Four-panel diagnostic plot ─────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# Panel A: Actual vs Predicted (test)
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(p_test, y_test, alpha=0.35, s=15, color='steelblue', label='Test firms')
lim = max(p_test.max(), y_test.max()) * 1.05
ax1.plot([0, lim], [0, lim], 'r--', lw=1.5, label='45° line')
ax1.set_xlabel('Predicted inflows (next Q)')
ax1.set_ylabel('Actual inflows (next Q)')
ax1.set_title(f'A: Actual vs Predicted\nTest R²={r2_score(y_test, p_test):.3f}')
ax1.legend(fontsize=8)
ax1.set_xlim(0, lim); ax1.set_ylim(0, lim)

# Panel B: Residuals vs Predicted (test)
ax2 = fig.add_subplot(gs[0, 1])
resid_test = y_test - p_test
ax2.scatter(p_test, resid_test, alpha=0.35, s=15, color='darkorange')
ax2.axhline(0, color='red', lw=1.5, linestyle='--')
ax2.axhline(resid_test.mean(), color='grey', lw=1, linestyle=':', label=f'Mean residual={resid_test.mean():.1f}')
ax2.set_xlabel('Predicted inflows')
ax2.set_ylabel('Residual (actual − predicted)')
ax2.set_title('B: Residual Plot (Test)')
ax2.legend(fontsize=8)

# Panel C: R² across splits (bar)
ax3 = fig.add_subplot(gs[1, 0])
splits = ['Train', 'Val', 'Test']
r2s    = []
for X, y in [(X_train, y_train), (X_val, y_val), (X_test, y_test)]:
    p = np.maximum(xgb_model.predict(X), 0)
    r2s.append(r2_score(y, p))
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax3.bar(splits, r2s, color=colors, width=0.5)
ax3.set_ylim(0, 1.0)
ax3.set_ylabel('R²')
ax3.set_title('C: R² by Split\n(gap = overfitting)')
for bar, val in zip(bars, r2s):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax3.axhline(0.5, color='red', lw=1, linestyle='--', alpha=0.5, label='R²=0.5 reference')
ax3.legend(fontsize=8)

# Panel D: Residual histogram (test)
ax4 = fig.add_subplot(gs[1, 1])
ax4.hist(resid_test, bins=40, color='mediumpurple', edgecolor='white', alpha=0.8)
ax4.axvline(0, color='red', lw=1.5, linestyle='--', label='Zero bias')
ax4.axvline(resid_test.mean(), color='black', lw=1.5, linestyle=':', label=f'Mean={resid_test.mean():.1f}')
ax4.set_xlabel('Residual (actual − predicted)')
ax4.set_ylabel('Count')
ax4.set_title('D: Residual Distribution (Test)')
ax4.legend(fontsize=8)

plt.suptitle('XGBoost Model Accuracy Diagnostics', fontsize=13, fontweight='bold', y=1.01)
plt.savefig(OUTPUT_DIR / 'model_accuracy_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved model_accuracy_diagnostics.png")

## 7  SHAP Analysis

Compute SHAP values on the test set to understand which features drive low/high predicted inflows.
These weights are the foundation of the posting-level ghost score.

In [ ]:
import shap

# Sample for speed (SHAP can be slow on large sets)
N_SHAP = min(2000, len(X_test))
rng = np.random.default_rng(42)
shap_idx = rng.choice(len(X_test), size=N_SHAP, replace=False)
X_shap = X_test[shap_idx]

print(f"Computing SHAP values on {N_SHAP} test samples...")
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_shap)
print("Done.")

# ── Beeswarm plot ──────────────────────────────────────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_COLS, show=False)
plt.title('SHAP Values — Impact on Predicted Inflows')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Bar plot (mean |SHAP|) ─────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP|)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Store mean |SHAP| per feature
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance = (
    pl.DataFrame({'feature': FEATURE_COLS, 'mean_abs_shap': mean_abs_shap})
    .sort('mean_abs_shap', descending=True)
)
print("\nTop features by mean |SHAP|:")
display(shap_importance)

## 8  Three Measures Comparison

For each company, aggregate across all available quarters:

| Measure | Formula |
|---------|---------|
| `avg_postings_per_quarter` | mean(num_postings) |
| `actual_fill_rate` | sum(actual inflows) / sum(num_postings) |
| `predicted_fill_rate` | sum(predicted inflows) / sum(num_postings) |

In [ ]:
# ── Generate predictions for ALL firm-quarters (train + val + test) ────────
eval_df = pl.concat([val_df, test_df])
X_eval, _ = to_xy(eval_df)
pred_eval = np.maximum(xgb_model.predict(X_eval), 0)
predictions = eval_df.with_columns(
    pl.Series('predicted_inflows', pred_eval)
)

# Save predictions
(
    predictions
    .select(['rcid', 'company', 'ticker', 'quarter',
             'num_postings', 'actual_inflows_next_q', 'predicted_inflows'])
    .write_parquet(OUTPUT_DIR / 'model_predictions.parquet')
)
print("Saved model_predictions.parquet")

# ── Aggregate to company level ─────────────────────────────────────────────
company_measures = (
    predictions
    .group_by(['rcid', 'company', 'ticker'])
    .agg([
        pl.col('num_postings').mean().alias('avg_postings_per_quarter'),
        (
            pl.col('actual_inflows_next_q').sum()
            / pl.col('num_postings').sum()
        ).alias('actual_fill_rate'),
        (
            pl.col('predicted_inflows').sum()
            / pl.col('num_postings').sum()
        ).alias('predicted_fill_rate'),
        pl.len().alias('quarters_observed'),
    ])
    .sort('predicted_fill_rate')
)

print(f"\nCompany measures table: {len(company_measures):,} companies")
display(company_measures.describe())

In [ ]:
# ── Top 20 most ghost-like (lowest predicted fill rate) ───────────────────
SHOW_COLS = ['company', 'ticker', 'avg_postings_per_quarter',
             'actual_fill_rate', 'predicted_fill_rate', 'quarters_observed']

print("=" * 65)
print("TOP 20 — MOST GHOST-LIKE (lowest predicted fill rate)")
print("=" * 65)
display(company_measures.head(20).select(SHOW_COLS))

print("\n" + "=" * 65)
print("BOTTOM 20 — HIGHEST PREDICTED FILL RATE (genuine hirers)")
print("=" * 65)
display(
    company_measures
    .sort('predicted_fill_rate', descending=True)
    .head(20)
    .select(SHOW_COLS)
)

# Save
company_measures.write_parquet(OUTPUT_DIR / 'company_measures.parquet')
print("\nSaved company_measures.parquet")

## 9  Posting-Level Ghost Score

**Threshold**: Use the 21st percentile of `predicted_fill_rate` to flag ghost companies
(~21% of postings are ghost per Ng 2024).

**Posting ghost score**: Normalized predicted inflows within each firm.
Score = 0 → this firm-quarter converts well; Score = 1 → very ghost-like.

In [ ]:
# ── Company-level ghost flag ───────────────────────────────────────────────
GHOST_THRESHOLD_PCT = 21.0  # adjustable

fill_rates = company_measures['predicted_fill_rate'].drop_nulls().to_numpy()
ghost_threshold = np.percentile(fill_rates, GHOST_THRESHOLD_PCT)

print(f"Ghost threshold (bottom {GHOST_THRESHOLD_PCT}th percentile): {ghost_threshold:.4f}")

company_measures = company_measures.with_columns(
    (pl.col('predicted_fill_rate') <= ghost_threshold).alias('is_ghost')
)
n_ghost = company_measures['is_ghost'].sum()
print(f"Companies flagged as ghost: {n_ghost:,} / {len(company_measures):,} "
      f"({n_ghost/len(company_measures)*100:.1f}%)")

# ── Firm-quarter ghost score (relative to firm's median prediction) ────────
firm_median = (
    predictions
    .group_by('rcid')
    .agg(pl.col('predicted_inflows').median().alias('firm_median_pred'))
)

ghost_scores_fq = (
    predictions
    .join(firm_median, on='rcid', how='left')
    .with_columns(
        (
            1.0 - pl.col('predicted_inflows')
            / pl.col('firm_median_pred').clip(lower_bound=0.01)
        )
        .clip(lower_bound=0.0, upper_bound=1.0)
        .alias('ghost_score')
    )
    .join(company_measures.select(['rcid', 'is_ghost']), on='rcid', how='left')
)

# Save the ghost_scores_fq DataFrame
ghost_scores_fq.write_parquet(OUTPUT_DIR / 'ghost_scores_fq.parquet')
print(f"Saved ghost_scores_fq.parquet  ({len(ghost_scores_fq):,} rows)")

print(f"\nPosting-level ghost score distribution:")
display(ghost_scores_fq['ghost_score'].describe())
print(f"\nMean ghost score for ghost firms    : "
      f"{ghost_scores_fq.filter(pl.col('is_ghost'))['ghost_score'].mean():.4f}")
print(f"Mean ghost score for non-ghost firms: "
      f"{ghost_scores_fq.filter(~pl.col('is_ghost'))['ghost_score'].mean():.4f}")

## 10  GMM Validation

Run Gaussian Mixture Model on the **raw** behavioral + company feature space (no PCA — only ~13 features).
**Goal**: Check whether ghost-heavy companies cluster together, validating that ghost behavior is systematic.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler as Scaler

# ── Company-level feature averages for GMM ────────────────────────────────
GMM_COLS = [
    'num_postings', 'avg_duration', 'still_active_rate',
    'salary_disclosure_rate', 'avg_salary', 'role_diversity', 'remote_rate',
    'headcount', 'headcount_growth_4q', 'log_headcount',
]

gmm_input = (
    predictions
    .group_by(['rcid', 'company', 'ticker'])
    .agg([pl.col(c).mean().alias(c) for c in GMM_COLS]
         + [pl.col('predicted_inflows').mean().alias('predicted_fill_rate_mean')])
    .join(company_measures.select(['rcid', 'is_ghost', 'predicted_fill_rate']),
          on='rcid', how='left')
    .drop_nulls(subset=GMM_COLS)
)

print(f"Firms in GMM: {len(gmm_input):,} (dropped {len(company_measures) - len(gmm_input):,} due to null features)")

X_gmm = gmm_input.select(GMM_COLS).fill_null(0).to_numpy()
scaler_gmm = Scaler()
X_gmm_s = scaler_gmm.fit_transform(X_gmm)

print(f"GMM input: {X_gmm_s.shape} (companies × features)")
print("Testing k=2 through k=8...")

bic_scores, aic_scores, gmm_models = [], [], {}
for k in range(2, 9):
    g = GaussianMixture(n_components=k, covariance_type='full',
                        random_state=42, n_init=5, max_iter=300)
    g.fit(X_gmm_s)
    bic_scores.append(g.bic(X_gmm_s))
    aic_scores.append(g.aic(X_gmm_s))
    gmm_models[k] = g
    print(f"  k={k}  BIC={g.bic(X_gmm_s):,.0f}  AIC={g.aic(X_gmm_s):,.0f}")

best_k = int(np.argmin(bic_scores)) + 2
print(f"\nBest k by BIC: {best_k}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(2, 9), bic_scores, 'o-', label='BIC')
ax.plot(range(2, 9), aic_scores, 's--', label='AIC')
ax.axvline(best_k, color='red', linestyle=':', lw=1.5, label=f'Best k={best_k}')
ax.set_xlabel('Number of clusters')
ax.set_ylabel('Information Criterion')
ax.set_title('GMM Model Selection (BIC/AIC)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gmm_selection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fit final GMM and profile clusters ────────────────────────────────────
CHOSEN_K = best_k   # or override: CHOSEN_K = 4

gmm_final   = gmm_models[CHOSEN_K]
cluster_lbl = gmm_final.predict(X_gmm_s)

gmm_result = gmm_input.with_columns(pl.Series('cluster', cluster_lbl))

# Cluster profiles
profile_cols = GMM_COLS + ['predicted_fill_rate']
cluster_profiles = (
    gmm_result
    .group_by('cluster')
    .agg(
        [pl.col(c).mean().round(4).alias(c) for c in profile_cols]
        + [pl.len().alias('n_companies'),
           (pl.col('is_ghost').cast(pl.Int8).mean()).alias('ghost_rate')]
    )
    .sort('cluster')
)
print("Cluster profiles:")
display(cluster_profiles)

# Cross-tab: cluster vs ghost flag
xtab = (
    gmm_result
    .group_by(['cluster', 'is_ghost'])
    .len()
    .sort(['cluster', 'is_ghost'])
)
print("\nCross-tab (cluster × ghost flag):")
display(xtab)

# ── Heatmap ───────────────────────────────────────────────────────────────
profile_arr = cluster_profiles.select(profile_cols).to_numpy()
norm_arr    = (profile_arr - profile_arr.mean(0)) / (profile_arr.std(0) + 1e-9)

fig, ax = plt.subplots(figsize=(13, max(3, CHOSEN_K + 1)))
sns.heatmap(
    norm_arr, annot=True, fmt='.2f', cmap='RdYlGn',
    xticklabels=profile_cols,
    yticklabels=[f'Cluster {i}' for i in range(CHOSEN_K)],
    ax=ax, linewidths=0.3,
)
ax.set_title('GMM Cluster Profiles (z-scored feature means)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gmm_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 11  Outputs & Visualization

In [ ]:
# ── Plot 1: Actual vs Predicted (test set) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
n_plot = min(3000, len(y_test))
ax.scatter(y_test[:n_plot], xgb_test_pred[:n_plot],
           alpha=0.3, s=8, rasterized=True, color='steelblue')
vmax = max(float(y_test.max()), float(xgb_test_pred.max()))
ax.plot([0, vmax], [0, vmax], 'r--', lw=1.5, label='Perfect')
ax.set_xlabel('Actual next-quarter inflows')
ax.set_ylabel('Predicted next-quarter inflows')
ax.set_title(f'Actual vs Predicted (Test)\nR²={r2_score(y_test, xgb_test_pred):.3f}  '
             f'MAE={mean_absolute_error(y_test, xgb_test_pred):.1f}')
ax.legend()

# ── Plot 2: Ghost score distribution ──────────────────────────────────────
ax = axes[1]
gs_vals = ghost_scores_fq['ghost_score'].drop_nulls().to_numpy()
ax.hist(gs_vals, bins=60, color='coral', edgecolor='black', linewidth=0.4)
ax.axvline(gs_vals.mean(), color='navy', linestyle='--',
           label=f'Mean = {gs_vals.mean():.2f}')
threshold_gs = np.percentile(gs_vals, 100 - GHOST_THRESHOLD_PCT)
ax.axvline(threshold_gs, color='red', linestyle=':',
           label=f'Ghost threshold = {threshold_gs:.2f}')
ax.set_xlabel('Ghost Score (0 = real hire, 1 = ghost)')
ax.set_ylabel('Firm-quarters')
ax.set_title(f'Ghost Score Distribution ({GHOST_THRESHOLD_PCT}% flagged)')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_and_ghost_score.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Three measures for top & bottom 20 companies ──────────────────
for grp, df_grp, fname in [
    ('Most Ghost-Like (lowest predicted fill rate)',
     company_measures.head(20).sort('predicted_fill_rate'),
     'top20_ghost'),
    ('Least Ghost-Like (highest predicted fill rate)',
     company_measures.sort('predicted_fill_rate', descending=True).head(20)
                     .sort('predicted_fill_rate', descending=True),
     'bottom20_ghost'),
]:
    labels  = df_grp['ticker'].to_list()
    x       = np.arange(len(labels))
    w       = 0.35

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(x - w/2, df_grp['actual_fill_rate'].to_list(),    w, label='Actual fill rate',    color='#2196F3')
    ax.barh(x + w/2, df_grp['predicted_fill_rate'].to_list(), w, label='Predicted fill rate', color='#FF5722')
    ax.set_yticks(x)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Fill Rate  (hires / open postings)')
    ax.set_title(f'{grp}\nActual vs Predicted Fill Rate')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Firms analyzed             : {len(company_measures):,}")
print(f"Ghost-flagged firms        : {company_measures['is_ghost'].sum():,} "
      f"({GHOST_THRESHOLD_PCT:.0f}% threshold)")
print(f"Firm-quarters in matrix    : {len(feature_matrix):,}")
print(f"Quarter range              : {feature_matrix['quarter'].min()} "
      f"→ {feature_matrix['quarter'].max()}")
print()
print("XGBoost performance:")
for split, y, p in [('Train', y_train, np.maximum(xgb_model.predict(X_train),0)),
                    ('Val',   y_val,   np.maximum(xgb_model.predict(X_val),  0)),
                    ('Test',  y_test,  xgb_test_pred)]:
    print(f"  {split:<6}  R²={r2_score(y,p):+.4f}  MAE={mean_absolute_error(y,p):.2f}")
print()
print("Files saved to", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f"  {f.name}")

In [ ]:
# ── Pipeline summary ──────────────────────────────────────────────────────
print(f"All outputs saved to: {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob('*')):
    if not f.name.startswith('.'):
        print(f"  {f.name}")


In [ ]:
# (Drive is mounted in the Setup cell above — nothing to do here)


In [ ]:
# ── Run the backtest from this notebook ───────────────────────────────────
import sys
sys.path.insert(0, str(OUTPUT_DIR))  # so Python can find the scripts

from ghost_backtest_v2 import main as run_backtest

ghost_file = str(OUTPUT_DIR / 'ghost_scores_fq.parquet')
nav, metrics = run_backtest(
    ghost_file  = ghost_file,
    start_date  = '2019-01-01',
    output_dir  = str(OUTPUT_DIR / 'backtest_output'),
)
print(metrics)


In [ ]:
# ── Concentration sensitivity sweep ───────────────────────────────────────
from ghost_concentration_compare_3 import run_comparison

results = run_comparison(
    ghost_file     = str(OUTPUT_DIR / 'ghost_scores_fq.parquet'),
    start_date     = '2019-01-01',
    concentrations = [0.05, 0.10, 0.25],
    output_dir     = str(OUTPUT_DIR / 'backtest_output'),
)
